In [1]:
# path: scripts/critical_load_generator_fixed.py
import random
import pandas as pd
import numpy as np
import os

NUM_SEQUENCES = 665

bp = "../../../data/simulation/"
CSV_OUTPUT_PATH = os.path.join(bp, "engine_critical_load_data.csv")
X_OUTPUT_PATH   = os.path.join(bp, "engine_critical_load_X.npy")
Y_OUTPUT_PATH   = os.path.join(bp, "engine_critical_load_y.npy")  # ragged per-sequence labels

# Ranges/params
RPM_MIN, RPM_MAX = 4500.0, 9000.0
TEMP_MIN, TEMP_MAX = 115.0, 145.0
EDGE = 120.0
RPM_EPS = 1e-6
TEMP_MARGIN = 0.05
EPS = 1e-6
NUDGE = 1e-4
EDGE_RPM_WOBBLE = 5.0
EDGE_TEMP_WOBBLE = 0.25

rand_open = lambda lo, hi: lo + (hi - lo) * random.random()

def bounded_step(rpm, delta):
    lo = RPM_MIN + RPM_EPS
    hi = RPM_MAX - RPM_EPS
    delta = min(delta, hi - rpm) if delta > 0 else max(delta, lo - rpm)
    new_rpm = rpm + delta
    if new_rpm <= lo + EDGE_RPM_WOBBLE: new_rpm = lo + EDGE_RPM_WOBBLE
    if new_rpm >= hi - EDGE_RPM_WOBBLE: new_rpm = hi - EDGE_RPM_WOBBLE
    return new_rpm, new_rpm - rpm

def pick_state_with_edges(desired, rpm):
    lo = RPM_MIN + RPM_EPS
    hi = RPM_MAX - RPM_EPS
    if rpm >= hi - EDGE: return "decel"
    if rpm <= lo + EDGE: return "accel"
    return desired

def sample_delta(state):
    if state == "accel": return random.uniform(40.0, 120.0)
    if state == "decel": return -random.uniform(40.0, 120.0)
    return random.uniform(-39.999999, 39.999999)

def temp_from_rpm(rpm):
    frac = (rpm - RPM_MIN) / (RPM_MAX - RPM_MIN)
    span = (TEMP_MAX - TEMP_MIN) - 2.0 * TEMP_MARGIN
    return (TEMP_MIN + TEMP_MARGIN) + span * frac

def simulate_sequence_30s():
    R = rand_open(RPM_MIN + RPM_EPS, RPM_MAX - RPM_EPS)
    rpm, temp, pres, vib, labels = [], [], [], [], []
    steps_left = 30
    while steps_left > 0:
        run_len = min(int(random.uniform(1, steps_left + 1)), steps_left)
        t = int(random.uniform(0, 3))  # 0 decel, 1 accel, 2 steady
        base_state = "decel" if t == 0 else ("accel" if t == 1 else "steady")
        for _ in range(run_len):
            state = pick_state_with_edges(base_state, R)
            d = sample_delta(state)
            R_new, d_eff = bounded_step(R, d)
            T_base = temp_from_rpm(R_new)
            transient = (d_eff / 120.0) * 0.4
            head_low, head_high = (T_base - TEMP_MIN), (TEMP_MAX - T_base)
            transient = min(transient, head_high - EPS) if transient > 0 else max(transient, -head_low + EPS)
            rem_low, rem_high = head_low + transient, head_high - transient
            jitter_mag = min(max(0.0, min(rem_low, rem_high) - EPS), 0.25)
            T = T_base + transient + random.uniform(-jitter_mag, jitter_mag)
            if T <= TEMP_MIN + NUDGE:
                T = TEMP_MIN + NUDGE + random.uniform(0.0, EDGE_TEMP_WOBBLE)
            elif T >= TEMP_MAX - NUDGE:
                T = TEMP_MAX - NUDGE - random.uniform(0.0, EDGE_TEMP_WOBBLE)
            P = 0.97 + random.uniform(-0.05, 0.05)
            V = 0.20 + (R_new - 5000.0) / 15000.0 + random.uniform(-0.08, 0.12)
            rpm.append(R_new); temp.append(T); pres.append(P); vib.append(V)
            labels.append(
                "CriticalLoad (accelerating)" if state == "accel" else (
                "CriticalLoad (decelerating)" if state == "decel" else "CriticalLoad (idle)"))
            R = R_new
        steps_left -= run_len
    X_seq = np.stack([temp, pres, rpm, vib], axis=1).astype(np.float32)
    return X_seq, labels  # labels is list[str] length 30

# ===== generate & save =====
feature_cols = ['Temperature', 'Pressure', 'RPM', 'Vibration']
rows, X_list, y_list = [], [], []
for seq_id in range(NUM_SEQUENCES):
    X_seq, lab_seq = simulate_sequence_30s()
    X_list.append(X_seq)
    y_list.append(np.array(lab_seq, dtype=object))  # per-sequence labels
    for t in range(30):
        rows.append({
            'Sequence': seq_id, 'Time': t + 1,
            'Temperature': float(X_seq[t, 0]), 'Pressure': float(X_seq[t, 1]),
            'RPM': float(X_seq[t, 2]), 'Vibration': float(X_seq[t, 3]),
            'State': lab_seq[t]
        })

os.makedirs(bp, exist_ok=True)
df = pd.DataFrame(rows, columns=['Sequence', 'Time', *feature_cols, 'State'])
X = np.stack(X_list, axis=0)          # (N,30,4)
y = np.array(y_list, dtype=object)    # (N,) object; each element length 30

df.to_csv(CSV_OUTPUT_PATH, index=False)
np.save(X_OUTPUT_PATH, X)
np.save(Y_OUTPUT_PATH, y)
print(f"X: {X.shape} | y: {y.shape} | CSV rows: {len(df)}")


X: (665, 30, 4) | y: (665, 30) | CSV rows: 19950
